## Simulador de Propriedades PVT para Engenharia Química
### Comparativo de Equações de Estado: RK vs SRK vs PR
**Desenvolvido por:** Matheus Wenner- Engenharia Química
---
*Este notebook realiza o cálculo do fator de compressibilidade (Z) para hidrocarbonetos leves e contaminantes, 
comparando modelos cúbicos fundamentais para a indústria de óleo e gás.*

In [1]:
# 1
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import json
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, clear_output

In [2]:
# 2 Garante que o gráfico apareça dentro do VS Code
pio.renderers.default = "vscode"

In [3]:
# 3 Tenta carregar o arquivo externo que criamos
try:
    with open('fluidos.json', 'r') as f:
        fluidos = json.load(f)
    print(f"✅ Banco de dados carregado: {len(fluidos)} componentes prontos.")
except FileNotFoundError:
    print("❌ Erro: Certifique-se de que o arquivo 'fluidos.json' está na mesma pasta!")

✅ Banco de dados carregado: 13 componentes prontos.


In [4]:
# 4
caixa_diagnostico = widgets.Output()

In [5]:
def calcular_modelos_puros(fluido_nome, T_c):
    p = fluidos[fluido_nome]
    T = T_c + 273.15
    R = 8.314
    Tr = T / p["Tc"]
    w = p["omega"]
    
    pressões_mpa = np.linspace(0.1, 30, 100)
    z_rk, z_srk, z_pr = [], [], []

    for P_mpa in pressões_mpa:
        P = P_mpa * 1e6
        
        # --- REDLICH-KWONG (RK) ---
        # Não utiliza o fator acêntrico (omega)
        a_rk = 0.42748 * (R**2 * p["Tc"]**2.5) / (p["Pc"] * np.sqrt(T))
        b_rk = 0.08664 * (R * p["Tc"]) / p["Pc"]
        A_rk, B_rk = (a_rk * P)/(R**2 * T**2), (b_rk * P)/(R * T)
        coef_rk = [1, -1, (A_rk - B_rk - B_rk**2), -A_rk * B_rk]
        z_rk.append(np.max(np.roots(coef_rk).real))

        # --- SOAVE-REDLICH-KWONG (SRK) ---
        # Introduz a correção alpha baseada no fator acêntrico
        m_srk = 0.480 + 1.574*w - 0.176*w**2
        alpha_srk = (1 + m_srk * (1 - np.sqrt(Tr)))**2
        a_srk = (0.42748 * R**2 * p["Tc"]**2 / p["Pc"]) * alpha_srk
        b_srk = 0.08664 * R * p["Tc"] / p["Pc"]
        A_srk, B_srk = (a_srk * P)/(R**2 * T**2), (b_srk * P)/(R * T)
        coef_srk = [1, -1, (A_srk - B_srk - B_srk**2), -A_srk * B_srk]
        z_srk.append(np.max(np.roots(coef_srk).real))

        # --- PENG-ROBINSON (PR) ---
        # Refina os coeficientes para melhor predição de densidade
        m_pr = 0.37464 + 1.54226*w - 0.26992*w**2
        alpha_pr = (1 + m_pr * (1 - np.sqrt(Tr)))**2
        a_pr = (0.45724 * R**2 * p["Tc"]**2 / p["Pc"]) * alpha_pr
        b_pr = 0.07780 * R * p["Tc"] / p["Pc"]
        A_pr, B_pr = (a_pr * P)/(R**2 * T**2), (b_pr * P)/(R * T)
        coef_pr = [1, -(1-B_pr), (A_pr - 2*B_pr - 3*B_pr**2), -(A_pr*B_pr - B_pr**2 - B_pr**3)]
        z_pr.append(np.max(np.roots(coef_pr).real))
        
    return pressões_mpa, z_rk, z_srk, z_pr

In [6]:
# --- BLOCO FINAL: GRÁFICO (PROTAGONISTA) + DIAGNÓSTICO + TEORIA Z ---

# 1. Espaço reservado para o diagnóstico dinâmico
caixa_diagnostico = widgets.Output()

# 2. Caixa Markdown fixa com a teoria sobre o Fator Z
teoria_z = widgets.HTML("""
<div style="background-color: #eef2f7; padding: 20px; border-radius: 10px; margin-top: 20px; font-family: serif; border-left: 5px solid #2c3e50;">
    <h2 style="color: #2c3e50; margin-top: 0;">📘 Fundamentos: O Fator de Compressibilidade (Z)</h2>
    <p>O fator <b>Z</b> quantifica o desvio de um gás real em relação ao comportamento de um <b>Gás Ideal</b> (onde Z = 1).</p>
    <ul>
        <li><b>Z < 1 (Domínio de Atração):</b> As forças atrativas de Van der Waals reduzem a distância entre as moléculas. O volume real é menor que o ideal. Comum em pressões moderadas.</li>
        <li><b>Z > 1 (Domínio de Repulsão):</b> As moléculas estão tão próximas que o seu volume físico (co-volume) impede maior compressão. As forças repulsivas dominam. Comum em pressões muito elevadas.</li>
    </ul>
    <p style="font-size: 0.9em; color: #555;"><i>Matematicamente: Z = PV / RT. Para gases ideais, PV = RT, logo Z = 1.</i></p>
</div>
""")

def plotar_comparativo(fluido_nome, T_c):
    # Chama o motor de cálculo
    p_eixo, z_rk, z_srk, z_pr = calcular_modelos_puros(fluido_nome, T_c)
    
    # 3. GERAÇÃO DO GRÁFICO (O ESPETÁCULO)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=p_eixo, y=z_rk, name='RK (Redlich-Kwong)', line=dict(color='red', width=2)))
    fig.add_trace(go.Scatter(x=p_eixo, y=z_srk, name='SRK (Soave-RK)', line=dict(color='green', width=2)))
    fig.add_trace(go.Scatter(x=p_eixo, y=z_pr, name='PR (Peng-Robinson)', line=dict(color='blue', width=3)))
    fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.3)
    
    fig.update_layout(
        title=dict(text=f"Análise PVT: {fluido_nome} a {T_c}°C", x=0.5, font=dict(size=20)),
        xaxis_title="Pressão (MPa)", yaxis_title="Fator de Compressibilidade (Z)",
        template="simple_white", margin=dict(t=80, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    # 4. LÓGICA DO DIAGNÓSTICO DINÂMICO
    z_final = z_pr[-1]
    if z_final < 1:
        titulo, cor_hex = "🟡 DOMÍNIO DE ATRAÇÃO", "#d4a017"
        exp = "As forças intermoleculares de atração predominam nesta faixa."
    else:
        titulo, cor_hex = "🔴 DOMÍNIO DE REPULSÃO", "#b22222"
        exp = "O efeito de co-volume e repulsão molecular domina a alta pressão."

    with caixa_diagnostico:
        clear_output()
        display(widgets.HTML(f"""
            <div style="border: 2px solid {cor_hex}; padding: 15px; border-radius: 8px; background-color: #fffcf5;">
                <b style="color: {cor_hex};">{titulo}:</b> {exp}
            </div>
        """))

    fig.show()



In [7]:
# 5. EXIBIÇÃO NA ORDEM SOLICITADA
# Gráfico primeiro, depois o diagnóstico, depois a teoria fixa
interact(plotar_comparativo, 
         fluido_nome=widgets.Dropdown(options=list(fluidos.keys()), description='Fluido:'),
         T_c=widgets.IntSlider(min=-100, max=500, step=5, value=25, description='Temp (°C):'))

display(caixa_diagnostico)
display(teoria_z)

interactive(children=(Dropdown(description='Fluido:', options=('metano', 'etano', 'propano', 'n-butano', 'i-bu…

Output()

HTML(value='\n<div style="background-color: #eef2f7; padding: 20px; border-radius: 10px; margin-top: 20px; fon…